# 🩺 Medical Reasoning LLM — QLoRA Training
## Fine-tuning Qwen2.5-3B-Instruct on Medical Chain-of-Thought QA

| Config | Value |
|---|---|
| Base model | Qwen/Qwen2.5-3B-Instruct |
| Dataset | FreedomIntelligence/medical-o1-reasoning-SFT |
| Method | QLoRA (4-bit NF4, LoRA r=16) |
| Training samples | 5,000 |
| Hardware | T4 GPU (Google Colab) |
| Est. time | ~4 hours |

### Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Run cells top to bottom
3. Each cell is independent and labelled

> ⚠️ This model is a research prototype — not for clinical use.

## 🔧 Cell 1 — Environment Check

In [ ]:
import subprocess, sys, os

# ── GPU check ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    print(f"  GPU      : {gpu_info}")
except Exception:
    print("  ⚠️  No GPU detected. Change runtime to T4 GPU.")
    print("  Runtime → Change runtime type → GPU → T4")

print(f"  Python   : {sys.version.split()[0]}")

# ── Disk space ────────────────────────────────────────────────────────────────
try:
    disk = subprocess.check_output(["df", "-h", "/"]).decode().split("\n")[1].split()
    print(f"  Disk     : {disk[3]} free / {disk[1]} total")
except Exception:
    pass

# ── Colab detection ───────────────────────────────────────────────────────────
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"  Colab    : {IS_COLAB}")
print("=" * 60)

## 📦 Cell 2 — Install Dependencies
*(~3 minutes on first run)*

In [ ]:
%%capture install_output
!pip install \
    torch==2.1.2 \
    transformers==4.43.4 \
    tokenizers==0.19.1 \
    accelerate==0.30.1 \
    datasets==2.19.2 \
    peft==0.11.1 \
    trl==0.9.4 \
    bitsandbytes==0.43.1 \
    rouge-score==0.1.2 \
    evaluate==0.4.2 \
    safetensors==0.4.3 \
    huggingface_hub==0.23.4 \
    tqdm rich pyyaml python-dotenv -q

import transformers, peft, trl, bitsandbytes, datasets
print("✓ All packages installed")
print(f"  transformers : {transformers.__version__}")
print(f"  peft         : {peft.__version__}")
print(f"  trl          : {trl.__version__}")
print(f"  bitsandbytes : {bitsandbytes.__version__}")
print(f"  datasets     : {datasets.__version__}")

## 💾 Cell 3 — Mount Google Drive *(Optional but Recommended)*
Saves checkpoints to Drive so you don't lose them if the Colab session disconnects.

In [ ]:
MOUNT_DRIVE = True   # ← Set to False to skip

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/medical-reasoning-llm/results"
        print(f"✓ Drive mounted. Outputs → {OUTPUT_DIR}")
    except Exception as e:
        print(f"Drive mount failed: {e}")
        MOUNT_DRIVE = False

if not MOUNT_DRIVE:
    OUTPUT_DIR = "/content/results"
    print(f"Using local Colab storage: {OUTPUT_DIR}")
    print("⚠️  Session data will be lost when Colab disconnects.")

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")

## ⚙️ Cell 4 — Configuration
All hyperparameters in one place. Edit here if needed.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN      = ""   # Optional: set if needed for private models
                     # Or: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME     = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES      = 5_000   # Use 25_000 for full dataset (needs ~16h on T4)
TRAIN_RATIO      = 0.80
VAL_RATIO        = 0.10
TEST_RATIO       = 0.10
MAX_SEQ_LENGTH   = 2048
SEED             = 42

# ── QLoRA ─────────────────────────────────────────────────────────────────────
LORA_R           = 16
LORA_ALPHA       = 32      # Always 2× r
LORA_DROPOUT     = 0.05
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS       = 3
BATCH_SIZE       = 2        # Per-device; T4 max with seq_len=2048
GRAD_ACCUM       = 4        # Effective batch = BATCH_SIZE × GRAD_ACCUM = 8
LEARNING_RATE    = 2e-4
LR_SCHEDULER     = "cosine"
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
LOGGING_STEPS    = 25
EVAL_STEPS       = 100
SAVE_STEPS       = 200
FP16             = True

EFFECTIVE_BATCH  = BATCH_SIZE * GRAD_ACCUM
APPROX_STEPS     = (int(NUM_SAMPLES * TRAIN_RATIO) // EFFECTIVE_BATCH) * NUM_EPOCHS

print("=" * 55)
print("  TRAINING CONFIGURATION")
print("=" * 55)
print(f"  Model         : {MODEL_NAME}")
print(f"  Dataset       : {DATASET_NAME}")
print(f"  Samples       : {NUM_SAMPLES:,}")
print(f"  LoRA r/alpha  : {LORA_R} / {LORA_ALPHA}  (scaling={LORA_ALPHA/LORA_R})")
print(f"  Epochs        : {NUM_EPOCHS}")
print(f"  Effective batch: {EFFECTIVE_BATCH}")
print(f"  Learning rate  : {LEARNING_RATE:.1e}")
print(f"  Approx steps   : ~{APPROX_STEPS:,}")
print(f"  Max seq length : {MAX_SEQ_LENGTH}")
print(f"  Output dir     : {OUTPUT_DIR}")
print("=" * 55)

##  Cell 5 — Load & Split Dataset

In [ ]:
import random
from datasets import load_dataset, DatasetDict

print(f"Loading {DATASET_NAME}...")
full_ds = load_dataset(DATASET_NAME, split="train")
print(f"Full dataset: {len(full_ds):,} samples")

# Draw reproducible subset
random.seed(SEED)
indices = sorted(random.sample(range(len(full_ds)), NUM_SAMPLES))
subset = full_ds.select(indices)

# Split
shuffled = subset.shuffle(seed=SEED)
n = len(shuffled)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
n_test  = n - n_train - n_val

splits = DatasetDict({
    "train"      : shuffled.select(range(0, n_train)),
    "validation" : shuffled.select(range(n_train, n_train + n_val)),
    "test"       : shuffled.select(range(n_train + n_val, n)),
})

print(f"\nSplits → train: {len(splits['train']):,} | "
      f"val: {len(splits['validation']):,} | test: {len(splits['test']):,}")

# Quick sanity check
sample = splits["train"][0]
print(f"\nSample question (first 120 chars):")
print(f"  {sample['Question'][:120]}...")

##  Cell 6 — Load Tokenizer & Format Dataset

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded")
print(f"  vocab_size : {tokenizer.vocab_size:,}")
print(f"  pad_token  : {tokenizer.pad_token!r}")
print(f"  eos_token  : {tokenizer.eos_token!r}")

# ── System message ────────────────────────────────────────────────────────────
SYSTEM_MESSAGE = (
    "You are an expert physician with deep knowledge of internal medicine, "
    "cardiology, neurology, and emergency medicine. When presented with a "
    "clinical scenario, you reason through it systematically before providing "
    "your final answer. Your reasoning should include: symptom analysis, "
    "relevant anatomy and physiology, differential diagnosis with reasoning, "
    "and evidence-based management. Always separate your thinking process "
    "from your final answer."
)

# ── Format function ───────────────────────────────────────────────────────────
def format_example(row):
    assistant_content = (
        "Let me reason through this step by step:\n"
        f"{row['Complex_CoT']}\n\n"
        "Final Answer:\n"
        f"{row['Response']}"
    )
    messages = [
        {"role": "system",    "content": SYSTEM_MESSAGE},
        {"role": "user",      "content": row["Question"]},
        {"role": "assistant", "content": assistant_content},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

# ── Apply to train and val splits ────────────────────────────────────────────
print("\nApplying chat template...")
formatted = {}
for split_name in ["train", "validation"]:
    ds = splits[split_name].map(format_example, num_proc=2, desc=f"Formatting {split_name}")
    before = len(ds)
    ds = ds.filter(
        lambda x: len(tokenizer.encode(x["text"], add_special_tokens=False)) <= MAX_SEQ_LENGTH,
        desc="Filtering long sequences"
    )
    print(f"  {split_name}: {before} → {len(ds)} (filtered {before - len(ds)} long sequences)")
    formatted[split_name] = ds

# Preview one formatted example
sample_text = formatted["train"][0]["text"]
sample_tokens = len(tokenizer.encode(sample_text))
print(f"\nFormatted example ({sample_tokens} tokens):")
print(sample_text[:600])
print("..." if len(sample_text) > 600 else "")

##  Cell 7 — Load Base Model (4-bit NF4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Loading base model with 4-bit NF4 quantization...")
print("(This downloads ~6 GB on first run, then uses the cache)\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,    # nested quantization: extra 0.4 bits/param saved
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache       = False   # disable KV cache during training
model.config.pretraining_tp  = 1       # needed for gradient flow with QLoRA

# Memory report
total_params     = sum(p.numel() for p in model.parameters())
mem_allocated_gb = torch.cuda.memory_allocated() / 1024**3

print(f"✓ Model loaded")
print(f"  Parameters     : {total_params/1e9:.2f}B")
print(f"  GPU memory used: {mem_allocated_gb:.2f} GB")
print(f"  Device map     : {model.hf_device_map}")

##  Cell 8 — Attach LoRA Adapters

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# Step 1: Prepare frozen quantized base for gradient flow
model = prepare_model_for_kbit_training(model)

# Step 2: Enable gradient checkpointing (trades compute for VRAM)
model.enable_input_require_grads()

# Step 3: Inject LoRA adapter layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)

# Parameter summary
total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen     = total - trainable

print("=" * 55)
print("  PARAMETER SUMMARY")
print("=" * 55)
print(f"  Trainable (LoRA adapters) : {trainable:>12,}  ({100*trainable/total:.4f}%)")
print(f"  Frozen    (quantized base): {frozen:>12,}  ({100*frozen/total:.4f}%)")
print(f"  Total                     : {total:>12,}")
print("=" * 55)
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, scaling={LORA_ALPHA/LORA_R:.1f}")
print(f"  Targets: {TARGET_MODULES}")

##  Cell 9 — Training Callbacks

In [ ]:
import time
from transformers import TrainerCallback, TrainerControl, TrainerState, TrainingArguments

# ── Fixed clinical question used to monitor training quality ─────────────────
MONITOR_QUESTION = (
    "A 58-year-old male with a 20-pack-year smoking history presents with "
    "3 weeks of worsening exertional dyspnea, orthopnea, and bilateral leg edema. "
    "JVP is elevated at 10 cm. Chest X-ray shows cardiomegaly and bilateral pleural "
    "effusions. BNP is 1,450 pg/mL. Echo shows EF 25%. What is the diagnosis and "
    "initial management?"
)

class SampleGenerationCallback(TrainerCallback):
    """Generates a clinical example at each eval to watch reasoning improve."""

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        step = state.global_step
        print(f"\n{'='*65}")
        print(f"  SAMPLE GENERATION — Step {step}")
        print(f"{'='*65}")

        messages = [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user",   "content": MONITOR_QUESTION},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt",
                           add_special_tokens=False).to(model.device)

        model.eval()
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        model.train()

        new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
        output   = tokenizer.decode(new_toks, skip_special_tokens=True)

        print(f"Q: {MONITOR_QUESTION[:100]}...")
        print(f"{'─'*65}")
        print(output[:800])
        print()

        # Log to file
        with open(f"{OUTPUT_DIR}/sample_generations.txt", "a") as f:
            f.write(f"\n{'='*65}\nSTEP {step} | {time.strftime('%H:%M:%S')}\n")
            f.write(f"Q: {MONITOR_QUESTION}\n\nA:\n{output}\n")

print("✓ SampleGenerationCallback defined")

##  Cell 10 — Setup SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"

sft_config = SFTConfig(
    # ── Paths ──────────────────────────────────────────────────────────────
    output_dir            = CHECKPOINT_DIR,

    # ── Core training ──────────────────────────────────────────────────────
    num_train_epochs      = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate         = LEARNING_RATE,
    lr_scheduler_type     = LR_SCHEDULER,
    warmup_ratio          = WARMUP_RATIO,
    weight_decay          = WEIGHT_DECAY,
    max_grad_norm         = 1.0,

    # ── Precision ──────────────────────────────────────────────────────────
    fp16                  = FP16,
    bf16                  = False,     # set True for A100

    # ── Optimizer ──────────────────────────────────────────────────────────
    optim                 = "paged_adamw_32bit",   # memory-efficient

    # ── Logging & checkpoints ──────────────────────────────────────────────
    logging_steps         = LOGGING_STEPS,
    eval_strategy         = "steps",
    eval_steps            = EVAL_STEPS,
    save_strategy         = "steps",
    save_steps            = SAVE_STEPS,
    save_total_limit      = 3,
    load_best_model_at_end= True,
    metric_for_best_model = "eval_loss",
    greater_is_better     = False,

    # ── SFT-specific ───────────────────────────────────────────────────────
    max_seq_length        = MAX_SEQ_LENGTH,
    dataset_text_field    = "text",
    packing               = False,

    # ── Misc ───────────────────────────────────────────────────────────────
    remove_unused_columns = False,
    report_to             = "none",   # change to "wandb" if tracking
    dataloader_num_workers= 2,
)

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset= formatted["train"],
    eval_dataset = formatted["validation"],
    args         = sft_config,
    callbacks    = [SampleGenerationCallback()],
)

train_count = len(formatted["train"])
val_count   = len(formatted["validation"])
steps_per_epoch = train_count // EFFECTIVE_BATCH
total_steps     = steps_per_epoch * NUM_EPOCHS

print("=" * 55)
print("  TRAINER READY")
print("=" * 55)
print(f"  Train samples   : {train_count:,}")
print(f"  Val   samples   : {val_count:,}")
print(f"  Steps / epoch   : ~{steps_per_epoch:,}")
print(f"  Total steps     : ~{total_steps:,}")
print(f"  Eval  every     : {EVAL_STEPS} steps")
print(f"  Save  every     : {SAVE_STEPS} steps")
print(f"  Checkpoint dir  : {CHECKPOINT_DIR}")
print("=" * 55)

##  Cell 11 — TRAIN
⏱️ *Estimated time: ~4 hours on T4*

Watch loss drop and the qualitative sample improve every 100 steps.

In [ ]:
print("Starting training...")
print("  Tip: Every 100 steps a clinical scenario is generated")
print("  so you can watch reasoning quality improve live.\n")

start_time = time.time()
train_result = trainer.train()
elapsed_min  = (time.time() - start_time) / 60

print("\n" + "=" * 55)
print("  TRAINING COMPLETE")
print("=" * 55)
print(f"  Train loss    : {train_result.training_loss:.4f}")
print(f"  Total steps   : {train_result.global_step}")
print(f"  Runtime (min) : {elapsed_min:.1f}")
print(f"  Runtime (hr)  : {elapsed_min/60:.2f}")
print("=" * 55)

##  Cell 12 — Save Adapter

In [ ]:
import os

# Save LoRA adapter only (~100 MB, not the full 6 GB model)
print(f"Saving adapter to: {ADAPTER_DIR}")
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Report saved files
print("\nSaved files:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(f"{ADAPTER_DIR}/{f}") / 1024**2
    print(f"  {f:<45} {size_mb:>7.2f} MB")

# ── Optional: push to HuggingFace Hub ────────────────────────────────────────
PUSH_TO_HUB  = False   # ← Set True after training to share publicly
HF_REPO_ID   = "YOUR_USERNAME/medical-reasoning-qwen2.5-3b-qlora"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"\n✓ Pushed to HuggingFace Hub: {HF_REPO_ID}")

print("\n✓ Adapter saved.")
print(f"  Reload anytime with:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{ADAPTER_DIR}')")

##  Cell 13 — Quick Evaluation on Test Set
*Computes accuracy and ROUGE-L on a 50-sample mini-test.*

In [ ]:
from rouge_score import rouge_scorer as rouge_lib

def extract_answer(text):
    """Parse final answer from generated output."""
    if "Final Answer:" in text:
        return text.split("Final Answer:", 1)[-1].strip()
    return text.strip().split("\n\n")[-1].strip()

def extract_reasoning(text):
    """Parse reasoning chain from generated output."""
    if "Let me reason" in text and "Final Answer:" in text:
        chain = text.split("Let me reason", 1)[-1]
        chain = chain.split("Final Answer:", 1)[0]
        return chain.strip()
    return text.strip()

print("Running quick eval on 50 test examples...")
test_sample = splits["test"].select(range(50))

scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)

pred_answers, gold_answers, pred_chains, gold_chains = [], [], [], []

model.eval()
for row in test_sample:
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": row["Question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output = tokenizer.decode(new_toks, skip_special_tokens=True)

    pred_answers.append(extract_answer(output).lower().strip())
    gold_answers.append(row["Response"].lower().strip())
    pred_chains.append(extract_reasoning(output))
    gold_chains.append(row["Complex_CoT"])

# Accuracy (exact match, normalized)
n = len(pred_answers)
correct = sum(p == g for p, g in zip(pred_answers, gold_answers))
accuracy = correct / n

# ROUGE-L
rouge_scores = [
    scorer.score(g, p)["rougeL"].fmeasure
    for p, g in zip(pred_chains, gold_chains)
]
avg_rouge_l = sum(rouge_scores) / len(rouge_scores)

print("\n" + "=" * 55)
print("  QUICK EVAL RESULTS  (50 test examples)")
print("=" * 55)
print(f"  Answer accuracy  : {accuracy:.4f}  ({correct}/{n})")
print(f"  ROUGE-L (CoT)    : {avg_rouge_l:.4f}")
print("=" * 55)
print("\nRun full eval with: python scripts/evaluate.py --compare_base")

##  Cell 14 — Inference Demo
Ask the trained model any clinical question.

In [ ]:
DEMO_QUESTIONS = [
    (
        "A 28-year-old woman presents with 3 days of progressive severe headache, "
        "photophobia, neck stiffness and fever of 39.2°C. She has no rash. "
        "What is the most likely diagnosis and immediate management?"
    ),
    (
        "A 72-year-old male with known COPD and 50 pack-year history presents "
        "with unintentional weight loss of 8 kg over 3 months, hemoptysis, "
        "and a new right upper lobe mass on CT chest. What is the most likely "
        "diagnosis and what investigations would you order?"
    ),
    (
        "A 45-year-old woman presents with fatigue, cold intolerance, weight gain, "
        "constipation, and dry skin for 6 months. Her TSH is 18 mIU/L, "
        "free T4 is 0.4 ng/dL. What is the diagnosis and treatment?"
    ),
]

def ask_model(question, max_new_tokens=600):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output   = tokenizer.decode(new_toks, skip_special_tokens=True)
    tokens   = len(new_toks)
    return output, tokens, elapsed

print("=" * 70)
print("  MEDICAL REASONING LLM — INFERENCE DEMO")
print("=" * 70)

for i, question in enumerate(DEMO_QUESTIONS, 1):
    output, n_tokens, t = ask_model(question)
    reasoning = extract_reasoning(output)
    answer    = extract_answer(output)

    print(f"\n{'─'*70}")
    print(f"  QUESTION {i}  ({n_tokens} tokens | {t:.1f}s)")
    print(f"{'─'*70}")
    print(f"Q: {question[:130]}...")
    print(f"\n[REASONING CHAIN]")
    print(reasoning[:800])
    print(f"\n[FINAL ANSWER]")
    print(answer[:400])

print(f"\n{'─'*70}")
print("⚠️  RESEARCH PROTOTYPE — Not for clinical use.")
print(f"{'─'*70}")